In [ ]:
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
import pyproj
from matplotlib.colors import LinearSegmentedColormap
from scipy.signal import convolve2d
from scipy.optimize import minimize_scalar
from scipy.ndimage import map_coordinates
from ipyleaflet import Map, ImageOverlay, CircleMarker
from ipywidgets import Output, VBox
from PIL import Image

In [ ]:
with rasterio.open("../data/Sonoma_DTM_2022-OAEC-0.01-degree.tif") as file:
    elevation = file.read(1)
    transform = file.transform
    crs = file.crs
    left, bottom, right, top = file.bounds

In [ ]:
water = gpd.read_file("../data/oaec-water.gpkg").to_crs(crs)
roads = gpd.read_file("../data/oaec-roads.gpkg").to_crs(crs)

In [ ]:
def point_in_disk(radius):
    width = 2 * radius + 1
    bigx, bigy = np.meshgrid(np.arange(10 * width), np.arange(10 * width))
    x = bigx / 10 - radius - 0.5
    y = bigy / 10 - radius - 0.5
    disk = x**2 + y**2 <= radius**2
    small = np.sum(np.sum(disk.reshape((width, 10, width, 10)), axis=-1), axis=-2)
    small[radius, radius] = 0
    small = -small / np.sum(small)
    small[radius, radius] = 1
    return small

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(8, 8))

for i, ax in enumerate(axs.flatten()):
    tmp = point_in_disk(i + 4)
    print(np.sum(tmp))
    tmp[tmp == 0] = np.nan
    ax.imshow(tmp, vmin=np.nanmin(tmp), vmax=0, cmap="Blues_r")

None

In [ ]:
def line_in_disk(radius, angle, linelength=20, linewidth=0.5):
    width = 2 * radius + 1
    bigx, bigy = np.meshgrid(np.arange(10 * width), np.arange(10 * width))
    x = bigx / 10 - radius - 0.5
    y = bigy / 10 - radius - 0.5
    disk = x**2 + y**2 <= radius**2
    minidisk = x**2 + y**2 <= (linelength / 2)**2
    line = np.abs(x * np.sin(-angle * np.pi/180) - y * np.cos(angle * np.pi/180)) < linewidth
    centerline = minidisk & line
    pill = disk & ~centerline

    shrink = lambda array: np.sum(np.sum(array.reshape((width, 10, width, 10)), axis=-1), axis=-2)
    small = shrink(pill)
    small_centerline = shrink(centerline)
    return -small / np.sum(small) + small_centerline / np.sum(small_centerline)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

tmp = line_in_disk(30, 30)
ax.imshow(tmp, vmin=np.nanmin(tmp), vmax=0, cmap="Blues_r")

None

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(8, 8))

for angle, ax in zip([0, 10, 20, 30, 45, 60, 80, 90, 135], axs.flatten()):
    tmp = line_in_disk(30, angle)
    print(np.sum(tmp))
    tmp[tmp == 0] = np.nan
    ax.imshow(tmp, vmin=np.nanmin(tmp), vmax=0, cmap="Blues_r")

None

In [ ]:
angles = {}

In [ ]:
angles[0] = convolve2d(elevation, line_in_disk(50, 0), mode="same", boundary="symm")

In [ ]:
angles[22.5] = convolve2d(elevation, line_in_disk(50, 22.5), mode="same", boundary="symm")

In [ ]:
angles[45] = convolve2d(elevation, line_in_disk(50, 45), mode="same", boundary="symm")

In [ ]:
angles[67.5] = convolve2d(elevation, line_in_disk(50, 67.5), mode="same", boundary="symm")

In [ ]:
angles[90] = convolve2d(elevation, line_in_disk(50, 90), mode="same", boundary="symm")

In [ ]:
angles[112.5] = convolve2d(elevation, line_in_disk(50, 112.5), mode="same", boundary="symm")

In [ ]:
angles[135] = convolve2d(elevation, line_in_disk(50, 135), mode="same", boundary="symm")

In [ ]:
angles[157.5] = convolve2d(elevation, line_in_disk(50, 157.5), mode="same", boundary="symm")

In [ ]:
np.savez("gully-OAEC-0.01-degree-25m-angular-convolutions.npz", **{f"angle-{angle}": array for angle, array in angles.items()})

In [ ]:
h = [0, 0.75, 0.25]       # horizontal is green
v = [0.25, 0.25, 1]       # vertical is blue

colormaps = {}
for angle, x, y in [
    (0, 1, 0),            # angle=0 (horizontal) is 1*h + 0*v
    (22.5, 0.75, 0.25),
    (45, 0.5, 0.5),
    (67.5, 0.25, 0.75),
    (90, 0, 1),
    (112.5, 0.25, 0.75),
    (135, 0.5, 0.5),
    (157.5, 0.75, 0.25),
]:
    colormaps[angle] = LinearSegmentedColormap(
        f"angle-{angle}",
        segmentdata={
            "red": [[0, x*h[0] + y*v[0], x*h[0] + y*v[0]], [1, 1, 1]],
            "green": [[0, x*h[1] + y*v[1], x*h[1] + y*v[1]], [1, 1, 1]],
            "blue": [[0, x*h[2] + y*v[2], x*h[2] + y*v[2]], [1, 1, 1]],
        },
        N=256,
    )

In [ ]:
allangles = np.concatenate([array[:, :, np.newaxis] for array in angles.values()], axis=-1)

In [ ]:
bestangle = np.argmin(allangles, axis=-1)

In [ ]:
MINIMUM = -5
MAXIMUM = -0.25

absolute = np.full(angles[0].shape, np.nan)
directed = np.full(angles[0].shape + (4,), np.nan)

In [ ]:
absolute = np.min(allangles, axis=-1)
absolute[absolute >= MAXIMUM] = np.nan

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))

ax.imshow(absolute, vmin=MINIMUM, vmax=MAXIMUM, cmap="gray", extent=(left, right, bottom, top))
water.plot(ax=ax, edgecolor="red", lw=2, alpha=0.25)

ax.set_xlabel("meters east")
ax.set_ylabel("meters north")

None

In [ ]:
for i, angle in enumerate(angles):
    mask = (bestangle == i) & (allangles[:, :, i] < MAXIMUM)
    directed[mask] = colormaps[angle](
        np.maximum(MINIMUM, (allangles[:, :, i][mask] - MINIMUM) / (MAXIMUM - MINIMUM))
    )

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))

ax.imshow(directed, extent=(left, right, bottom, top))
water.plot(ax=ax, edgecolor="red", lw=2, alpha=0.25)

ax.set_xlabel("meters east")
ax.set_ylabel("meters north")

None

In [ ]:
southwest = pyproj.Transformer.from_crs(crs, "EPSG:4326", always_xy=True).transform(left, bottom)
northeast = pyproj.Transformer.from_crs(crs, "EPSG:4326", always_xy=True).transform(right, top)

In [ ]:
color_image = 255 * (directed - np.nanmin(directed)) / (np.nanmax(directed) - np.nanmin(directed))
color_image[:, :, 3] = 256 - 255 * (absolute - np.nanmin(absolute)) / (np.nanmax(absolute) - np.nanmin(absolute))
color_image[np.isnan(color_image)] = 0
color_image = color_image.astype(np.uint8)
im = Image.fromarray(color_image, mode="RGBA")
with io.BytesIO() as output:
    im.save(output, format="PNG")
    png_data = output.getvalue()
data_url = f"data:image/png;base64,{base64.b64encode(png_data).decode('utf-8')}"

In [ ]:
LOCATION = (38.41278794816254, -122.95649975160721)

overlay = ImageOverlay(url=data_url, bounds=(southwest[::-1], northeast[::-1]))

circle = CircleMarker(location=LOCATION, radius=2, color="red")

m = Map(center=LOCATION, zoom=14)
m.add_layer(overlay)
m.add_layer(circle)
out = Output()

def handle_map_click(**kwargs):
    if kwargs.get("type") == "click":
        latlng = kwargs.get("coordinates")
        with out:
            circle.location = latlng
            print(f"longitude, latitude = {latlng[::-1]}")
m.on_interaction(handle_map_click)

display(VBox([m, out]))

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(8, 14))

longitude, latitude = [-122.95384526252748, 38.4146597775645]
col, row = map(int, ~transform * pyproj.Transformer.from_crs("EPSG:4326", crs, always_xy=True).transform(longitude, latitude))

local = elevation[
    max(0, row - 50):min(elevation.shape[0], row + 51),
    max(0, col - 50):min(elevation.shape[1], col + 51),
]
local = local[::-1, :]
assert local.shape == (101, 101)

minimum = minimize_scalar(lambda angle: np.sum(line_in_disk(50, angle, linelength=20) * local), bounds=(0, 180), method="bounded")
assert minimum.success
angle = minimum.x

x0 = 50 - 40*np.sin(angle * np.pi/180)
x1 = 50 + 40*np.sin(angle * np.pi/180)
y0 = 50 - 40*np.cos(angle * np.pi/180)
y1 = 50 + 40*np.cos(angle * np.pi/180)
length = np.sqrt((x1 - x0)**2 + (y1 - y0)**2)

contour = ax[0].contourf(local, cmap="grey")
ax[0].plot([x0, x1], [y0, y1], c="red", lw=2)
ax[0].set_xticks([0, 20, 40, 60, 80, 100], ["0", "10", "20", "30", "40", "50"])
ax[0].set_yticks([0, 20, 40, 60, 80, 100], ["0", "10", "20", "30", "40", "50"])
ax[0].set_xlabel("meters east")
ax[0].set_ylabel("meters north")
fig.colorbar(contour, ax=ax[0], label="elevation (meters)")

x_vals = np.linspace(x0, x1, 100)
y_vals = np.linspace(y0, y1, 100)
profile = map_coordinates(local, np.vstack([y_vals, x_vals]), order=1)
ax[1].plot(np.linspace(-length/4, length/4, len(profile)), profile, c="red")
ax[1].set_xlabel("position along red line (meters)")
ax[1].set_ylabel("elevation (meters)")

None